## Table of content
1.1 Libraries

1.2 Data Sets 

2. Aggregated order number mean for the complete data set.
3. Analyzing the results
4. Creating a loyalty flag.
5. Identifying spending habits of loyalty customers
6. Indentifying different spending types in customers
7. Creating an order frequency flag.
8. Exporting data frame

# Creating a new notebook.

## 1.1 Libraries

In [6]:
# Importing Libraries

In [8]:
import pandas as pd
import numpy as np
import os

## 2.1 Data Sets

In [11]:
# Creating a path

In [13]:
path=r'/Users/konstant/Documents/Achievement 4. Instacart Basket Analysis'

In [15]:
# Importing Data Sets

In [17]:
df_ords_prods_merge= pd.read_pickle(os.path.join(path, '02 Data', 'Prepared Data', 'orders_products_combined_new_variables.pkl'))

In [19]:
df_ords_prods_merge.shape

(32404859, 17)

# 2. Aggregated order number mean for the complete data set.

In [22]:
# Grouping department_id using the groupby() function, and performing a single aggregation (mean) on order_number. 

In [24]:
df_ords_prods_merge.groupby('department_id').agg({'order_number': ['mean']})

,order_number
,mean
department_id,
1,15.457838
2,17.277920
3,17.170395
4,17.811403
5,15.215751
6,16.439806
7,17.225802
8,15.340650


# 3. Analyzing the results

## The mean was calculated for this data set (32,404,859 rows) and a subset of this data (1,000,000 rows): 
## The number of departments matches (21 in both sets) 
## The mean on the larger data sets is slightly different from the mean on the subset accross all departments. Example:                     
## * department_id 1:   Full data set - 15.457838 / Sub-set - 15.577493
## * department_id 11:  Full data set - 16.170638 / Sub-set - 15.447411
## * department_id 21:  Full data set - 22.902379 / Sub-set - 25.535479

# 4. Creating a loyalty flag.

In [29]:
# To create the loyalty flag, we first need to create a new aggregated variable for each user to show their max orders:
# 1. Split the data into groups based on the “user_id” column.
# 2. Apply the transform() function on the “order_number” column to generate the maximum orders for each user.
# 3. Create a new column, “max_order,” into which the aggregation results will be placed.

In [31]:
df_ords_prods_merge['max_order'] = df_ords_prods_merge.groupby(['user_id'])['order_number'].transform(np.max)

/var/folders/_v/8zb8smns61xgjmbl859s2nwc0000gn/T/ipykernel_1239/136815596.py:1: FutureWarning: The provided callable <function max at 0x10663e5c0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  df_ords_prods_merge['max_order'] = df_ords_prods_merge.groupby(['user_id'])['order_number'].transform(np.max)


In [33]:
# checking that the column has been succesfully added. 

In [35]:
df_ords_prods_merge.head(2)

,product_id,product_name,aisle_id,department_id,prices,order_id,user_id,order_number,orders_day_of_week,hour_of_purchase,days_since_prior_order,add_to_cart_order,reordered,_merge,price_range_loc,busiest_days,busiest_period_of_day,max_order
0,1,Chocolate Sandwich Cookies,61,19,5.8,3139998,138,28,6,11,3.0,5,0,both,Mid-range product,Regular days,"Most orders,",32
1,1,Chocolate Sandwich Cookies,61,19,5.8,1977647,138,30,6,17,20.0,1,1,both,Mid-range product,Regular days,"Average orders,",32


In [37]:
# Finally adding a loyalty flag column by using the loc() function and 3 defined criteria for customer loyalty

In [39]:
df_ords_prods_merge.loc[df_ords_prods_merge['max_order'] > 40, 'loyalty_flag'] = 'Loyal customer'

In [41]:
df_ords_prods_merge.loc[(df_ords_prods_merge['max_order'] <= 40) & (df_ords_prods_merge['max_order'] > 10), 'loyalty_flag'] = 'Regular customer'

In [43]:
df_ords_prods_merge.loc[df_ords_prods_merge['max_order'] <= 10, 'loyalty_flag'] = 'New customer'

In [45]:
# checking that the loyalty flag column has been succesfully added

In [47]:
df_ords_prods_merge.head(2)

,product_id,product_name,aisle_id,department_id,prices,order_id,user_id,order_number,orders_day_of_week,hour_of_purchase,days_since_prior_order,add_to_cart_order,reordered,_merge,price_range_loc,busiest_days,busiest_period_of_day,max_order,loyalty_flag
0,1,Chocolate Sandwich Cookies,61,19,5.8,3139998,138,28,6,11,3.0,5,0,both,Mid-range product,Regular days,"Most orders,",32,Regular customer
1,1,Chocolate Sandwich Cookies,61,19,5.8,1977647,138,30,6,17,20.0,1,1,both,Mid-range product,Regular days,"Average orders,",32,Regular customer


In [49]:
# Checking the output of the column by means of frequency

In [51]:
df_ords_prods_merge['loyalty_flag'].value_counts(dropna=False)

loyalty_flag
Regular customer    15876776
Loyal customer      10284093
New customer         6243990
Name: count, dtype: int64

# 5. Identifying spending habits of loyalty customers

In [54]:
df_ords_prods_merge.groupby('loyalty_flag').agg({'prices': ['mean', 'min', 'max']})

prices              
                       mean  min      max
loyalty_flag                             
Loyal customer    10.386336  1.0  99999.0
New customer      13.294670  1.0  99999.0
Regular customer  12.495717  1.0  99999.0

## The average price of products purchased by loyal customers are the lowest, followed by regular customers. New customers spend the most on products. 

# 6. Indentifying different spending types in customers

In [58]:
# To create the spending flag, we first need to create a new aggregated variable for each user to show their max spend:

In [60]:
df_ords_prods_merge['max_spend'] = df_ords_prods_merge.groupby(['user_id'])['prices'].transform(np.max)

/var/folders/_v/8zb8smns61xgjmbl859s2nwc0000gn/T/ipykernel_1239/1340972696.py:1: FutureWarning: The provided callable <function max at 0x10663e5c0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  df_ords_prods_merge['max_spend'] = df_ords_prods_merge.groupby(['user_id'])['prices'].transform(np.max)


In [62]:
# checking that the column has been succesfully added.

In [64]:
df_ords_prods_merge.head(2)

,product_id,product_name,aisle_id,department_id,prices,order_id,user_id,order_number,orders_day_of_week,hour_of_purchase,days_since_prior_order,add_to_cart_order,reordered,_merge,price_range_loc,busiest_days,busiest_period_of_day,max_order,loyalty_flag,max_spend
0,1,Chocolate Sandwich Cookies,61,19,5.8,3139998,138,28,6,11,3.0,5,0,both,Mid-range product,Regular days,"Most orders,",32,Regular customer,20.0
1,1,Chocolate Sandwich Cookies,61,19,5.8,1977647,138,30,6,17,20.0,1,1,both,Mid-range product,Regular days,"Average orders,",32,Regular customer,20.0


In [66]:
# Now adding a spending flag column by using the loc() function and 3 defined criteria for customer loyalty

In [68]:
df_ords_prods_merge.loc[df_ords_prods_merge['max_spend'] < 10, 'spending_flag'] = 'Low spender'

In [70]:
df_ords_prods_merge.loc[df_ords_prods_merge['max_spend'] >= 10, 'spending_flag'] = 'High spender'

In [72]:
# checking that the spending flag column has been succesfully added

In [74]:
df_ords_prods_merge.head(2)

,product_id,product_name,aisle_id,department_id,prices,order_id,user_id,order_number,orders_day_of_week,hour_of_purchase,...,add_to_cart_order,reordered,_merge,price_range_loc,busiest_days,busiest_period_of_day,max_order,loyalty_flag,max_spend,spending_flag
0,1,Chocolate Sandwich Cookies,61,19,5.8,3139998,138,28,6,11,...,5,0,both,Mid-range product,Regular days,"Most orders,",32,Regular customer,20.0,High spender
1,1,Chocolate Sandwich Cookies,61,19,5.8,1977647,138,30,6,17,...,1,1,both,Mid-range product,Regular days,"Average orders,",32,Regular customer,20.0,High spender


In [76]:
# Checking the output of the column by means of frequency

In [78]:
df_ords_prods_merge['spending_flag'].value_counts(dropna=False)

spending_flag
High spender    32372969
Low spender        31890
Name: count, dtype: int64

# 7. Creating an order frequency flag.

In [81]:
# Aggregating the days_since_prior_order with the median.

In [83]:
df_ords_prods_merge['days_since_last_order_median'] = df_ords_prods_merge.groupby(['user_id'])['days_since_prior_order'].transform(np.median)

/var/folders/_v/8zb8smns61xgjmbl859s2nwc0000gn/T/ipykernel_1239/1651649641.py:1: FutureWarning: The provided callable <function median at 0x106781c60> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  df_ords_prods_merge['days_since_last_order_median'] = df_ords_prods_merge.groupby(['user_id'])['days_since_prior_order'].transform(np.median)


In [85]:
df_ords_prods_merge.head(2)

,product_id,product_name,aisle_id,department_id,prices,order_id,user_id,order_number,orders_day_of_week,hour_of_purchase,...,reordered,_merge,price_range_loc,busiest_days,busiest_period_of_day,max_order,loyalty_flag,max_spend,spending_flag,days_since_last_order_median
0,1,Chocolate Sandwich Cookies,61,19,5.8,3139998,138,28,6,11,...,0,both,Mid-range product,Regular days,"Most orders,",32,Regular customer,20.0,High spender,8.0
1,1,Chocolate Sandwich Cookies,61,19,5.8,1977647,138,30,6,17,...,1,both,Mid-range product,Regular days,"Average orders,",32,Regular customer,20.0,High spender,8.0


In [87]:
# Now setting up the criteria

In [89]:
df_ords_prods_merge.loc[df_ords_prods_merge['days_since_last_order_median'] > 20, 'order_frequency_flag'] = 'Non-frequent customer'

In [91]:
df_ords_prods_merge.loc[(df_ords_prods_merge['days_since_last_order_median'] <= 20) & (df_ords_prods_merge['days_since_last_order_median'] > 10), 'order_frequency_flag'] = 'Regular customer'

In [93]:
df_ords_prods_merge.loc[df_ords_prods_merge['days_since_last_order_median'] <= 10, 'order_frequency_flag'] = 'Frequent customer'

In [95]:
# checking that the spending flag column has been succesfully added

In [97]:
df_ords_prods_merge.head(2)

,product_id,product_name,aisle_id,department_id,prices,order_id,user_id,order_number,orders_day_of_week,hour_of_purchase,...,_merge,price_range_loc,busiest_days,busiest_period_of_day,max_order,loyalty_flag,max_spend,spending_flag,days_since_last_order_median,order_frequency_flag
0,1,Chocolate Sandwich Cookies,61,19,5.8,3139998,138,28,6,11,...,both,Mid-range product,Regular days,"Most orders,",32,Regular customer,20.0,High spender,8.0,Frequent customer
1,1,Chocolate Sandwich Cookies,61,19,5.8,1977647,138,30,6,17,...,both,Mid-range product,Regular days,"Average orders,",32,Regular customer,20.0,High spender,8.0,Frequent customer


In [99]:
# Checking the output of the column by means of frequency

In [101]:
df_ords_prods_merge['order_frequency_flag'].value_counts(dropna=False)

order_frequency_flag
Frequent customer        21559853
Regular customer          7208564
Non-frequent customer     3636437
NaN                             5
Name: count, dtype: int64

# 9. Exporting data frame as a pickle file.

In [104]:
df_ords_prods_merge.to_pickle(os.path.join(path, '02 Data','Prepared Data', 'ords_prods_merge_aggregated_variables.pkl'))
